# LeetCode 2903 — Sum of Values at Indices With K Set Bits bằng Python

Link đề: https://leetcode.com/problems/sum-of-values-at-indices-with-k-set-bits/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cộng các giá trị `nums[i]` mà **chỉ số `i`** (viết ở hệ nhị phân) có **đúng `k` bit bằng `1`**.
2. Hiểu **biểu diễn nhị phân**, khái niệm **set bit** (bit bằng `1`) và **popcount** (đếm số bit `1`).
3. Đi từ cách **đổi sang chuỗi nhị phân `bin(i)` rồi đếm ký tự `'1'`** → **`int.bit_count()`** → **vòng lặp dịch bit `& 1` và `>>`** → **mẹo Kernighan `i &= i - 1`** → **bảng quy hoạch động `pc[i] = pc[i >> 1] + (i & 1)`**.
4. Biết các toán tử bit cơ bản: `&`, `>>`, và vì sao `i & (i - 1)` xóa bit `1` thấp nhất.
5. Tránh các lỗi kinh điển: đếm bit của **giá trị** `nums[i]` thay vì **chỉ số** `i`, nhầm `bin(i)` có tiền tố `'0b'`, so sánh `>= k` thay vì `== k`, và vòng lặp vô hạn khi quên cập nhật `i`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (đánh chỉ số từ **0**) và số nguyên `k`. Hãy trả về **tổng** các phần tử `nums[i]` sao cho biểu diễn nhị phân của **chỉ số `i`** có **đúng `k` bit bằng `1`** (`set bit`).

**Ví dụ 1**

- Input: `nums = [5, 10, 1, 5, 2]`, `k = 1`
- Output: `13`
- Giải thích: biểu diễn nhị phân của các chỉ số: `0 = 000`, `1 = 001`, `2 = 010`, `3 = 011`, `4 = 100`. Các chỉ số `1`, `2`, `4` có đúng một bit `1`. Tổng `nums[1] + nums[2] + nums[4] = 10 + 1 + 2 = 13`.

**Ví dụ 2**

- Input: `nums = [4, 3, 2, 1]`, `k = 2`
- Output: `1`
- Giải thích: các chỉ số `0 = 00`, `1 = 01`, `2 = 10`, `3 = 11`. Chỉ có chỉ số `3` có đúng hai bit `1`, nên tổng là `nums[3] = 1`.

**Ràng buộc:** `1 <= nums.length <= 1000`; `1 <= nums[i] <= 10^5`; `0 <= k <= 10`.

In [1]:
from typing import List
import random
import sys

examples = [
    ([5, 10, 1, 5, 2], 1, 13),
    ([4, 3, 2, 1], 2, 1),
]
for nums, k, expected in examples:
    print("nums =", nums, "k =", k, "-> kỳ vọng", expected)
print("Python", sys.version.split()[0])

nums = [5, 10, 1, 5, 2] k = 1 -> kỳ vọng 13
nums = [4, 3, 2, 1] k = 2 -> kỳ vọng 1
Python 3.13.18


## 3. Kiến thức nền

### 3.1. Hệ nhị phân và `bin()`

Máy tính biểu diễn số bằng hệ **nhị phân** (chỉ có `0` và `1`). Ví dụ `5 = 101` vì `1·4 + 0·2 + 1·1`. Hàm `bin(x)` trả **chuỗi** nhị phân có tiền tố `'0b'`: `bin(5)` là `'0b101'`. Mỗi chữ số nhị phân gọi là một **bit**.

In [2]:
for x in [0, 1, 2, 3, 4, 5, 10]:
    print(x, "->", bin(x), "| bỏ tiền tố:", bin(x)[2:])

0 -> 0b0 | bỏ tiền tố: 0
1 -> 0b1 | bỏ tiền tố: 1
2 -> 0b10 | bỏ tiền tố: 10
3 -> 0b11 | bỏ tiền tố: 11
4 -> 0b100 | bỏ tiền tố: 100
5 -> 0b101 | bỏ tiền tố: 101
10 -> 0b1010 | bỏ tiền tố: 1010


### 3.2. Set bit và popcount

**Set bit** là bit có giá trị `1`. **Popcount** (population count) của một số là **số bit `1`** của nó. Ví dụ `popcount(5) = 2` (`101`), `popcount(8) = 1` (`1000`), `popcount(0) = 0`. Bài này cần tính popcount của **các chỉ số** `0, 1, ..., n-1`.

In [3]:
def popcount_str(x):
    return bin(x).count("1")        # đếm số ký tự '1' trong chuỗi nhị phân


for x in [0, 1, 5, 8, 15, 1023]:
    print(f"popcount({x}) =", popcount_str(x))

popcount(0) = 0
popcount(1) = 1
popcount(5) = 2
popcount(8) = 1
popcount(15) = 4
popcount(1023) = 10


### 3.3. `str.count` và tiền tố `'0b'` có làm sai không?

`bin(x).count("1")` đếm ký tự `'1'` trong **toàn chuỗi**, kể cả tiền tố `'0b'`. Tiền tố chỉ chứa `'0'` và `'b'`, **không có `'1'`**, nên kết quả vẫn đúng. Ta dùng `count("1")` (không phải `count("0")`, vì `'0b'` có một ký tự `'0'` sẽ làm sai).

In [4]:
print(bin(5).count("1"))        # 2: đúng
print(bin(5).count("0"))        # 2: gồm cả '0' của '0b' + '0' của '101' -> KHÔNG phải số bit 0 thật
print(bin(5)[2:].count("0"))    # 1: bỏ tiền tố mới đúng

2
2
1


### 3.4. Toán tử bit `&` và `>>`

- `x & 1`: lấy **bit thấp nhất** của `x` (bằng `1` nếu `x` lẻ).
- `x >> 1`: **dịch phải** một bit, tương đương `x // 2` (bỏ bit thấp nhất).

Hai thao tác này cho cách **đếm bit tay**: lặp lại "cộng bit thấp nhất rồi dịch phải" cho tới khi hết bit.

In [5]:
x = 11                                  # 1011
print(bin(x), x & 1, bin(x >> 1), x >> 1)     # bit thấp nhất = 1; x >> 1 = 101 = 5
count = 0
while x > 0:
    count += x & 1
    x >>= 1
print("popcount(11) =", count)          # 3

0b1011 1 0b101 5
popcount(11) = 3


### 3.5. Mẹo Kernighan: `x & (x - 1)` xóa bit `1` thấp nhất

Trừ 1 khỏi `x` làm bit `1` thấp nhất thành `0` và các bit `0` bên phải nó thành `1`. Phép `&` với `x` ban đầu **xóa đúng bit `1` thấp nhất**. Lặp lại tới khi `x = 0`, số lần lặp là **số bit `1`**: chỉ lặp bằng số bit `1`, không phải bằng độ dài bit.

In [6]:
x = 12                                    # 1100
steps = 0
while x:
    print(bin(x), "->", bin(x & (x - 1)))
    x &= x - 1
    steps += 1
print("số bước =", steps, "= popcount(12)")    # 2

0b1100 -> 0b1000
0b1000 -> 0b0
số bước = 2 = popcount(12)


### 3.6. `int.bit_count()` có sẵn

Từ Python 3.10, số nguyên có phương thức **`x.bit_count()`** trả popcount trực tiếp: `5 .bit_count()` là `2` (nếu viết literal số thì chừa dấu cách trước dấu chấm hoặc dùng biến). Nếu máy chấm dùng Python cũ hơn, hãy dùng `bin(x).count("1")`.

In [7]:
x = 5
print(x.bit_count())                       # 2
print((255).bit_count())                   # 8
print(all(i.bit_count() == bin(i).count("1") for i in range(5000)))    # True: hai cách luôn khớp

2
8
True


### 3.7. Bảng popcount bằng quy hoạch động: `pc[i] = pc[i >> 1] + (i & 1)`

Popcount của `i` bằng popcount của `i >> 1` (bỏ bit thấp nhất) cộng bit thấp nhất `i & 1`. Vì `i >> 1 < i`, ta tính bảng `pc[0..n-1]` từ nhỏ đến lớn trong `O(n)`.

In [8]:
n = 10
pc = [0] * n
for i in range(1, n):
    pc[i] = pc[i >> 1] + (i & 1)
print(pc)                                  # [0, 1, 1, 2, 1, 2, 2, 3, 1, 2]
print(pc == [bin(i).count("1") for i in range(n)])

[0, 1, 1, 2, 1, 2, 2, 3, 1, 2]
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bài hỏi về **chỉ số**, không phải giá trị. Hãy viết chỉ số ở dạng nhị phân rồi đếm bit `1`, ví dụ 1 (`nums = [5, 10, 1, 5, 2]`, `k = 1`):

| chỉ số `i` | nhị phân | popcount | `== k = 1`? | `nums[i]` | cộng? |
|---|---|---|---|---|---|
| 0 | `000` | 0 | không | 5 | không |
| 1 | `001` | 1 | **có** | 10 | **+10** |
| 2 | `010` | 1 | **có** | 1 | **+1** |
| 3 | `011` | 2 | không | 5 | không |
| 4 | `100` | 1 | **có** | 2 | **+2** |

Tổng `10 + 1 + 2 = 13`. Thuật toán: **với mỗi chỉ số `i`, tính popcount(`i`); nếu bằng `k` thì cộng `nums[i]`**.

## 5. Cách 1 — Brute Force / baseline: `bin(i).count("1")`

**Ý tưởng.** Duyệt `i` từ `0` đến `n - 1`, đổi `i` sang chuỗi nhị phân rồi đếm ký tự `'1'`; nếu bằng `k` thì cộng `nums[i]`. Đơn giản và rất dễ đọc.

In [9]:
def sum_bin_count(nums: List[int], k: int) -> int:
    total = 0
    for i in range(len(nums)):
        if bin(i).count("1") == k:         # số bit 1 của CHỈ SỐ i
            total += nums[i]
    return total


for nums, k, expected in examples:
    print(nums, k, "->", sum_bin_count(nums, k), "(kỳ vọng", expected, ")")

[5, 10, 1, 5, 2] 1 -> 13 (kỳ vọng 13 )
[4, 3, 2, 1] 2 -> 1 (kỳ vọng 1 )


In [10]:
# Quan sát từng chỉ số của ví dụ 1
nums, k = [5, 10, 1, 5, 2], 1
for i in range(len(nums)):
    print(f"i={i} bin={bin(i)[2:]:>3} popcount={bin(i).count('1')} {'<-- cộng ' + str(nums[i]) if bin(i).count('1') == k else ''}")

i=0 bin=  0 popcount=0 
i=1 bin=  1 popcount=1 <-- cộng 10
i=2 bin= 10 popcount=1 <-- cộng 1
i=3 bin= 11 popcount=2 
i=4 bin=100 popcount=1 <-- cộng 2


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài `nums`. Mỗi chỉ số `i` đổi sang nhị phân tốn `O(log i)`: **Time `O(n log n)`**; chỉ dùng vài biến (chuỗi nhị phân tạm dài `O(log n)`): **Space `O(1)`** (không kể chuỗi tạm nhỏ).

**Nút thắt.** Không có nút thắt đáng kể với `n <= 1000`. Ta chỉ muốn hiểu cách **đếm bit** theo nhiều cách khác nhau, và cách rẻ nhất để có popcount của **mọi** chỉ số liên tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

Mấu chốt của bài chỉ là **"popcount của chỉ số"**. Có nhiều cách tính popcount:

- đổi sang chuỗi rồi đếm `'1'` (Cách 1);
- dùng hàm có sẵn `i.bit_count()` (Cách 2);
- **tự dịch bit** `& 1`, `>>= 1` (Cách 3);
- **mẹo Kernighan** `i &= i - 1` chỉ lặp bằng số bit `1` (Cách 4);
- **tái sử dụng kết quả của chỉ số nhỏ hơn**: `pc[i] = pc[i >> 1] + (i & 1)` (Cách 5), tính popcount của mọi chỉ số `0..n-1` chỉ trong `O(n)` tổng cộng.

Phần cộng dồn `nums[i]` là như nhau ở mọi cách.

## 7. Các cách tiếp theo

### Cách 2 — `int.bit_count()`

**Giải thích.** `i.bit_count()` trả luôn số bit `1`. Ngắn nhất và nhanh nhất cho một lời gọi.

In [11]:
def sum_bit_count(nums: List[int], k: int) -> int:
    return sum(v for i, v in enumerate(nums) if i.bit_count() == k)


for nums, k, expected in examples:
    print(nums, k, "->", sum_bit_count(nums, k), "(kỳ vọng", expected, ")")

[5, 10, 1, 5, 2] 1 -> 13 (kỳ vọng 13 )
[4, 3, 2, 1] 2 -> 1 (kỳ vọng 1 )


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List

class Solution:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        return sum(v for i, v in enumerate(nums) if i.bit_count() == k)

**Độ phức tạp Cách 2:** **Time `O(n)`** (mỗi `bit_count()` là phép dựng sẵn cho số nhỏ, coi như `O(1)`), **Space `O(1)`**.

### Cách 3 — Tự dịch bit: `i & 1` rồi `i >>= 1`

**Giải thích.** Viết hàm `popcount_shift(i)`: lặp `while i` cộng `i & 1` rồi `i >>= 1`. Số vòng lặp bằng **độ dài nhị phân** của `i` (khoảng `log2 i`).

In [13]:
def popcount_shift(i: int) -> int:
    count = 0
    while i:
        count += i & 1          # bit thấp nhất (0 hoặc 1)
        i >>= 1                 # bỏ bit thấp nhất
    return count


def sum_shift(nums: List[int], k: int) -> int:
    return sum(nums[i] for i in range(len(nums)) if popcount_shift(i) == k)


for nums, k, expected in examples:
    print(nums, k, "->", sum_shift(nums, k), "(kỳ vọng", expected, ")")

[5, 10, 1, 5, 2] 1 -> 13 (kỳ vọng 13 )
[4, 3, 2, 1] 2 -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** **Time `O(n log n)`** (mỗi chỉ số `O(log n)` vòng), **Space `O(1)`**.

### Cách 4 — Mẹo Kernighan: `i &= i - 1`

**Giải thích.** Mỗi vòng xóa đúng một bit `1` (bit `1` thấp nhất), nên số vòng bằng **popcount**, không phải độ dài bit. Với chỉ số có ít bit `1`, nhanh hơn Cách 3.

In [14]:
def popcount_kernighan(i: int) -> int:
    count = 0
    while i:
        i &= i - 1              # xóa bit 1 thấp nhất
        count += 1
    return count


def sum_kernighan(nums: List[int], k: int) -> int:
    return sum(nums[i] for i in range(len(nums)) if popcount_kernighan(i) == k)


for nums, k, expected in examples:
    print(nums, k, "->", sum_kernighan(nums, k), "(kỳ vọng", expected, ")")

[5, 10, 1, 5, 2] 1 -> 13 (kỳ vọng 13 )
[4, 3, 2, 1] 2 -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n · b)`** với `b` là popcount lớn nhất (tối đa `log n`), **Space `O(1)`**.

### Cách 5 — Bảng popcount bằng quy hoạch động: `pc[i] = pc[i >> 1] + (i & 1)`

**Giải thích.** Tính một lần `pc[0..n-1]`: mỗi phần tử chỉ cần **một phép cộng** nhờ tái sử dụng `pc[i >> 1]`. Sau đó cộng `nums[i]` ở những `i` có `pc[i] == k`. Cách này không cần hàm đếm bit riêng cho từng chỉ số, và rất hữu ích khi phải tra popcount **nhiều lần**.

In [15]:
def sum_dp_table(nums: List[int], k: int) -> int:
    n = len(nums)
    pc = [0] * n                            # pc[i] = số bit 1 của i
    for i in range(1, n):
        pc[i] = pc[i >> 1] + (i & 1)
    return sum(nums[i] for i in range(n) if pc[i] == k)


for nums, k, expected in examples:
    print(nums, k, "->", sum_dp_table(nums, k), "(kỳ vọng", expected, ")")

[5, 10, 1, 5, 2] 1 -> 13 (kỳ vọng 13 )
[4, 3, 2, 1] 2 -> 1 (kỳ vọng 1 )


In [16]:
# Code nộp LeetCode cho Cách 5
from typing import List

class Solution:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        n = len(nums)
        pc = [0] * n
        for i in range(1, n):
            pc[i] = pc[i >> 1] + (i & 1)
        return sum(nums[i] for i in range(n) if pc[i] == k)

**Độ phức tạp Cách 5:** **Time `O(n)`** (một phép cộng cho mỗi chỉ số), **Space `O(n)`** cho bảng `pc`.

> Cách 2 là cách nên nộp (ngắn, nhanh, rõ). Các cách khác giúp bạn **hiểu popcount**; Cách 5 mở ra kỹ thuật "tính bảng từ kết quả nhỏ hơn" dùng rất nhiều trong bài bit.

## 8. TRACE từng bước

Ta theo dõi (a) Cách 4 đếm bit bằng `i &= i - 1` cho vài chỉ số; (b) Cách 5 xây bảng `pc` và cộng.

In [17]:
def trace_kernighan(i):
    x, steps = i, 0
    print(f"popcount({i}) = popcount({bin(i)})")
    while x:
        nxt = x & (x - 1)
        steps += 1
        print(f"  bước {steps}: x={bin(x):>8} x-1={bin(x - 1):>8} x&(x-1)={bin(nxt):>8}")
        x = nxt
    print(f"  => {steps} bit 1")
    return steps


assert trace_kernighan(0) == 0
assert trace_kernighan(8) == 1
assert trace_kernighan(13) == 3

popcount(0) = popcount(0b0)
  => 0 bit 1
popcount(8) = popcount(0b1000)
  bước 1: x=  0b1000 x-1=   0b111 x&(x-1)=     0b0
  => 1 bit 1
popcount(13) = popcount(0b1101)
  bước 1: x=  0b1101 x-1=  0b1100 x&(x-1)=  0b1100
  bước 2: x=  0b1100 x-1=  0b1011 x&(x-1)=  0b1000
  bước 3: x=  0b1000 x-1=   0b111 x&(x-1)=     0b0
  => 3 bit 1


In [18]:
def trace_dp(nums, k):
    n = len(nums)
    pc, total = [0] * n, 0
    print(f"nums={nums}, k={k}")
    for i in range(n):
        if i:
            pc[i] = pc[i >> 1] + (i & 1)
        take = pc[i] == k
        if take:
            total += nums[i]
        print(f"i={i} bin={bin(i)[2:]:>3} pc[{i}]=pc[{i >> 1}]+{i & 1}={pc[i]} -> {'cộng ' + str(nums[i]) + f', total={total}' if take else 'bỏ'}")
    print("tổng =", total)
    return total


assert trace_dp([5, 10, 1, 5, 2], 1) == 13

nums=[5, 10, 1, 5, 2], k=1
i=0 bin=  0 pc[0]=pc[0]+0=0 -> bỏ
i=1 bin=  1 pc[1]=pc[0]+1=1 -> cộng 10, total=10
i=2 bin= 10 pc[2]=pc[1]+0=1 -> cộng 1, total=11
i=3 bin= 11 pc[3]=pc[1]+1=2 -> bỏ
i=4 bin=100 pc[4]=pc[2]+0=1 -> cộng 2, total=13
tổng = 13


Bảng tóm tắt (Cách 5, `nums = [5, 10, 1, 5, 2]`, `k = 1`):

| `i` | nhị phân | `pc[i]` | `pc[i] == 1`? | `total` |
|---|---|---|---|---|
| 0 | `000` | 0 | không | 0 |
| 1 | `001` | 1 | có | 10 |
| 2 | `010` | 1 | có | 11 |
| 3 | `011` | 2 | không | 11 |
| 4 | `100` | 1 | có | 13 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (`bit_count`). Đúng class `Solution`, đúng method `sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int`. (Nếu máy chấm dùng Python < 3.10, thay `i.bit_count()` bằng `bin(i).count("1")`.)

In [19]:
from typing import List

class Solution:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        return sum(v for i, v in enumerate(nums) if i.bit_count() == k)


sol = Solution()
for nums, k, expected in examples:
    print(nums, k, "->", sol.sumIndicesWithKSetBits(nums, k))

[5, 10, 1, 5, 2] 1 -> 13
[4, 3, 2, 1] 2 -> 1


## 10. Vì sao thuật toán đúng

- **Đúng đối tượng:** đề xét **chỉ số** `i` (từ `0`), nên ta tính popcount của `i`, không phải của `nums[i]`. `enumerate(nums)` cho đúng cặp `(i, nums[i])`.
- **Đúng điều kiện:** cộng `nums[i]` khi và chỉ khi `popcount(i) == k` (đúng `k` bit `1`, không phải ít nhất `k`).
- **Mỗi chỉ số xét đúng một lần:** nên mỗi `nums[i]` được cộng tối đa một lần, và tổng là tổng đúng của tập chỉ số thỏa điều kiện.
- **Các cách đếm bit tương đương:** `bin(i).count("1")`, `bit_count`, dịch bit, Kernighan và bảng DP đều tính **cùng một hàm popcount** (đã kiểm bằng đối chiếu trong phần test). Với bảng DP: `i = 2·(i >> 1) + (i & 1)` nên số bit `1` của `i` là số bit `1` của `i >> 1` cộng bit thấp nhất.
- **Trường hợp `k = 0`:** chỉ có chỉ số `0` có popcount bằng `0`, nên đáp án là `nums[0]`.
- **Tính chất kiểm được:** đáp án luôn `<= sum(nums)`, và bằng `0` nếu không chỉ số nào có popcount `k` (ví dụ `k` lớn hơn `log2 n`).

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài `nums`.

- **Cách 1, 3:** Time `O(n log n)` (mỗi chỉ số `O(log n)`), Space `O(1)`.
- **Cách 2:** Time `O(n)` (popcount của số nhỏ coi như hằng số), Space `O(1)`.
- **Cách 4:** Time `O(n · b)` với `b <= log n`, Space `O(1)`.
- **Cách 5:** Time **`O(n)`** (một phép cộng cho mỗi chỉ số), Space `O(n)`.
- Mọi cách phải xét từng phần tử nên không thể tốt hơn `O(n)`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` rồi kiểm tra bằng `assert`.

In [20]:
sol = Solution()
impls = {
    "bin_count": sum_bin_count,
    "bit_count": sum_bit_count,
    "shift": sum_shift,
    "kernighan": sum_kernighan,
    "dp_table": sum_dp_table,
    "submit": sol.sumIndicesWithKSetBits,
}

for nums, k, expected in examples:
    for key, fn in impls.items():
        assert fn(list(nums), k) == expected, (key, nums, k, fn(list(nums), k), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [21]:
def check_all(nums, k, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums), k)
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([7], 0, 7, "một phần tử, k = 0: chỉ số 0 có 0 bit 1")
check_all([7], 1, 0, "một phần tử, k = 1: không có chỉ số nào")
check_all([3, 4], 1, 4, "hai phần tử: chỉ số 1 có 1 bit")
check_all([1, 2, 3, 4, 5, 6, 7, 8], 0, 1, "k = 0 chỉ lấy nums[0]")
check_all([1, 2, 3, 4, 5, 6, 7, 8], 2, 4 + 6 + 7, "k = 2: chỉ số 3, 5, 6")
check_all([1, 2, 3, 4, 5, 6, 7, 8], 3, 8, "k = 3: chỉ số 7")
check_all([5] * 16, 4, 5, "k = 4: chỉ số 15 duy nhất")
check_all([5] * 16, 10, 0, "k = 10 quá lớn: không có chỉ số nào")

PASS một phần tử, k = 0: chỉ số 0 có 0 bit 1 -> 7
PASS một phần tử, k = 1: không có chỉ số nào -> 0
PASS hai phần tử: chỉ số 1 có 1 bit -> 4
PASS k = 0 chỉ lấy nums[0] -> 1
PASS k = 2: chỉ số 3, 5, 6 -> 17
PASS k = 3: chỉ số 7 -> 8
PASS k = 4: chỉ số 15 duy nhất -> 5
PASS k = 10 quá lớn: không có chỉ số nào -> 0


### 12.2. Tính chất và phản ví dụ

In [22]:
# Tính chất 1: tổng theo mọi k bằng tổng toàn mảng (mỗi chỉ số có đúng một popcount)
random.seed(3)
for _ in range(100):
    n = random.randint(1, 200)
    nums = [random.randint(1, 100000) for _ in range(n)]
    total_by_k = sum(sol.sumIndicesWithKSetBits(nums, k) for k in range(0, 11))
    assert total_by_k == sum(nums)
print("PASS tổng theo mọi k = tổng toàn mảng (n <= 200 nên popcount <= 10)")

# Tính chất 2: kết quả không vượt quá tổng mảng, và k = 0 cho nums[0]
nums = [random.randint(1, 100) for _ in range(50)]
assert sol.sumIndicesWithKSetBits(nums, 0) == nums[0]
assert sol.sumIndicesWithKSetBits(nums, 5) <= sum(nums)
print("PASS k = 0 cho nums[0]; kết quả <= tổng")

# Đầu vào không bị sửa
nums = [5, 10, 1, 5, 2]
copy = list(nums)
for key, fn in impls.items():
    fn(nums, 1)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: đếm bit của GIÁ TRỊ nums[i] thay vì CHỈ SỐ i
def sai_dem_gia_tri(nums, k):
    return sum(v for v in nums if v.bit_count() == k)

assert sai_dem_gia_tri([5, 10, 1, 5, 2], 1) != 13
print("PASS phản ví dụ: đếm bit giá trị cho", sai_dem_gia_tri([5, 10, 1, 5, 2], 1), "thay vì 13")

# Phản ví dụ 2: dùng >= k thay vì == k
def sai_lon_hon(nums, k):
    return sum(v for i, v in enumerate(nums) if i.bit_count() >= k)

assert sai_lon_hon([5, 10, 1, 5, 2], 1) != 13
print("PASS phản ví dụ: >= k cho", sai_lon_hon([5, 10, 1, 5, 2], 1), "thay vì 13")

PASS tổng theo mọi k = tổng toàn mảng (n <= 200 nên popcount <= 10)
PASS k = 0 cho nums[0]; kết quả <= tổng
PASS đầu vào không bị sửa
PASS phản ví dụ: đếm bit giá trị cho 3 thay vì 13
PASS phản ví dụ: >= k cho 18 thay vì 13


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [23]:
random.seed(2903)
for trial in range(1000):
    n = random.randint(1, 1000)
    nums = [random.randint(1, 100000) for _ in range(n)]
    k = random.randint(0, 10)
    ref = sum_bin_count(nums, k)
    for key, fn in impls.items():
        assert fn(list(nums), k) == ref, (key, n, k)
print("PASS 1000 mảng ngẫu nhiên (n tới 1000) - mọi cách khớp nhau")

# Bảng popcount của các cách tính phải khớp trên miền chỉ số 0..4095
for i in range(4096):
    a = bin(i).count("1")
    assert i.bit_count() == a == popcount_shift(i) == popcount_kernighan(i)
print("PASS 4 cách đếm bit khớp nhau với mọi i < 4096")

PASS 1000 mảng ngẫu nhiên (n tới 1000) - mọi cách khớp nhau
PASS 4 cách đếm bit khớp nhau với mọi i < 4096


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm bit của giá trị `nums[i]` thay vì chỉ số `i`

In [24]:
def sai_gia_tri(nums, k):
    return sum(v for v in nums if v.bit_count() == k)


nums = [5, 10, 1, 5, 2]
print("sai  :", sai_gia_tri(nums, 1))                    # đếm bit của 5, 10, 1, 5, 2
print("đúng :", Solution().sumIndicesWithKSetBits(nums, 1))

sai  : 3
đúng : 13


**Vì sao sai:** đề nói "**chỉ số** có đúng `k` bit `1`". `v.bit_count()` đếm bit của **giá trị** `v`. Hãy dùng `enumerate` để có cả chỉ số `i` và giá trị `v`, rồi đếm bit của `i`.

### Lỗi 2: Dùng `>= k` hoặc `<= k` thay vì `== k`

In [25]:
def sai_so_sanh(nums, k):
    return sum(v for i, v in enumerate(nums) if i.bit_count() >= k)


print("sai  :", sai_so_sanh([5, 10, 1, 5, 2], 1))
print("đúng :", Solution().sumIndicesWithKSetBits([5, 10, 1, 5, 2], 1))

sai  : 18
đúng : 13


**Vì sao sai:** đề yêu cầu **đúng** `k` bit `1`. Với `k = 1` nhưng chỉ số `3 = 011` có hai bit `1` bị cộng nhầm nếu dùng `>=`.

### Lỗi 3: Đếm ký tự sai khi dùng `bin()`

In [26]:
def sai_dem_khong(nums, k):
    # tưởng k là số bit 0 và đếm '0' trong bin(i) -> tính cả '0' của tiền tố '0b'
    return sum(v for i, v in enumerate(nums) if bin(i).count("0") == k)


print("sai  :", sai_dem_khong([5, 10, 1, 5, 2], 1))
print("đúng :", Solution().sumIndicesWithKSetBits([5, 10, 1, 5, 2], 1))

sai  : 15
đúng : 13


**Vì sao sai:** đề đếm bit **`1`**, và `bin(i)` có tiền tố `'0b'` chứa một ký tự `'0'`. Đếm `'0'` vừa sai đối tượng vừa dính tiền tố. Luôn `count("1")` (tiền tố `'0b'` không chứa `'1'`) hoặc dùng `bit_count`.

### Lỗi 4: Vòng lặp vô hạn khi quên cập nhật số

In [27]:
def popcount_sai(i, max_steps=20):
    count, steps = 0, 0
    while i and steps < max_steps:          # max_steps chỉ để demo không treo máy
        count += i & 1
        # quên i >>= 1  -> i không bao giờ giảm
        steps += 1
    return count, steps


print("sai  :", popcount_sai(5))             # chạy tới max_steps, count phình ra
print("đúng :", popcount_shift(5))           # 2

sai  : (20, 20)
đúng : 2


**Vì sao sai:** vòng `while i:` chỉ dừng khi `i` về `0`. Nếu quên `i >>= 1` (hoặc `i &= i - 1`) thì `i` không đổi và vòng lặp **không bao giờ kết thúc** (ở trên ta thêm `max_steps` chỉ để minh họa).

### Lỗi 5: Nhầm chỉ số bắt đầu từ 1

In [28]:
def sai_chi_so_tu_1(nums, k):
    return sum(v for i, v in enumerate(nums, start=1) if i.bit_count() == k)


print("sai  :", sai_chi_so_tu_1([5, 10, 1, 5, 2], 1))        # chỉ số 1, 2, 3, 4, 5 thay vì 0..4
print("đúng :", Solution().sumIndicesWithKSetBits([5, 10, 1, 5, 2], 1))

sai  : 20
đúng : 13


**Vì sao sai:** đề nói mảng đánh chỉ số từ **0**. `enumerate(nums, start=1)` đẩy mọi chỉ số lên 1 đơn vị (nên chỉ số `3 = 11` thay vì `2 = 10`, v.v.). Dùng `enumerate(nums)` mặc định.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "số bit 1 / set bit / popcount" → nghĩ tới **`x.bit_count()`** hoặc `bin(x).count("1")`.
- Thấy "chỉ số" trong điều kiện → nhớ dùng `enumerate` và áp dụng điều kiện lên **`i`**.
- Cần popcount **cho mọi `0..n-1`** → bảng DP `pc[i] = pc[i >> 1] + (i & 1)`.
- Muốn lặp đúng số bit 1: **Kernighan** `x &= x - 1`.
- Kiểm tra nhanh bằng bảng: viết các chỉ số ở nhị phân và gạch chân.
- Kiểm chứng các cách đếm bit bằng vòng `for i in range(...)` và `assert`.

## 15. Biến thể của bài

1. Cộng `nums[i]` mà `popcount(i)` là **số nguyên tố** (hoặc là số chẵn).
2. Cộng các phần tử có popcount của **giá trị** bằng `k` (đổi đối tượng đếm).
3. Đếm **bao nhiêu chỉ số** có đúng `k` bit `1` trong `[0, n)`: dùng tổ hợp khi `n` là lũy thừa của 2 (`C(log2 n, k)`).
4. Tìm **chỉ số nhỏ nhất** có `k` bit `1` lớn hơn hoặc bằng `i` (liên quan hoán vị bit).
5. Cộng với các chỉ số có `popcount(i)` **lẻ** (parity).
6. Với `n` cực lớn (`10^9`): liệt kê các chỉ số có `k` bit `1` bằng `itertools.combinations` thay vì duyệt hết.

Ta thử biến thể 1, 3 và 6:

In [29]:
from math import comb
from itertools import combinations

def sum_popcount_prime(nums):
    primes = {2, 3, 5, 7}
    return sum(v for i, v in enumerate(nums) if i.bit_count() in primes)

print("1. popcount nguyên tố:", sum_popcount_prime(list(range(1, 33))))

n, k = 16, 2
print("3. đếm chỉ số có", k, "bit 1 trong [0, 16):", sum(1 for i in range(n) if i.bit_count() == k), "= C(4, 2) =", comb(4, 2))

# 6. liệt kê chỉ số có k bit 1 mà không duyệt hết: chọn vị trí các bit 1
def indices_with_k_bits(n, k):
    bits = n.bit_length()
    out = []
    for pos in combinations(range(bits), k):
        i = sum(1 << p for p in pos)
        if i < n:
            out.append(i)
    return sorted(out)

print("6. chỉ số có 2 bit 1 trong [0, 20):", indices_with_k_bits(20, 2))
assert indices_with_k_bits(20, 2) == [i for i in range(20) if i.bit_count() == 2]

1. popcount nguyên tố: 362
3. đếm chỉ số có 2 bit 1 trong [0, 16): 6 = C(4, 2) = 6
6. chỉ số có 2 bit 1 trong [0, 20): [3, 5, 6, 9, 10, 12, 17, 18]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `bin(i).count("1")` | đổi chuỗi rồi đếm | `O(n log n)` | `O(1)` | dễ đọc, hiểu bản chất |
| 2. `i.bit_count()` | hàm có sẵn (3.10+) | `O(n)` | `O(1)` | **cách nộp bài** |
| 3. Dịch bit `& 1`, `>>= 1` | đếm bit tay | `O(n log n)` | `O(1)` | học toán tử bit |
| 4. Kernighan `i &= i - 1` | lặp bằng số bit 1 | `O(n · b)` | `O(1)` | số bit 1 ít, phỏng vấn bit |
| 5. DP `pc[i] = pc[i>>1] + (i&1)` | tái sử dụng kết quả nhỏ hơn | `O(n)` | `O(n)` | cần popcount của mọi chỉ số liên tiếp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `bin(6)`, `bin(6).count("1")`, `6 & 1`, `6 >> 1`, `[i for i in range(8) if bin(i).count("1") == 2]`.

**Bài 2 (tính tay).** Với `nums = [3, 1, 4, 1, 5, 9, 2, 6]` và `k = 2`: viết nhị phân của chỉ số `0..7`, chọn các chỉ số có đúng 2 bit `1` và tính tổng.

**Bài 3 (điền TODO).** Hoàn thành `sum_todo(nums, k)` bằng bảng DP `pc`.

In [30]:
def sum_todo(nums, k):
    n = len(nums)
    pc = [0] * n
    # TODO 1: với i từ 1 đến n - 1: pc[i] = pc[i >> 1] + (i & 1)
    # TODO 2: return tổng nums[i] với mọi i có pc[i] == k
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `popcount_loop(x)` bằng vòng lặp `& 1` và `>>=` (Cách 3), rồi `sum_loop2(nums, k)` dùng nó. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_indices(n, k)` trả về **số chỉ số** `i` trong `[0, n)` có đúng `k` bit `1` mà **không** dùng vòng lặp qua mọi `i` khi `n` là lũy thừa của `2` (dùng `math.comb`). (b) Viết `sum_even_popcount(nums)` cộng các `nums[i]` mà `popcount(i)` là **số chẵn**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `'0b110'`, `2`, `0`, `3`, `[3, 5, 6]`. Bài 2: viết `0=000`, `1=001`, ..., `7=111`.
- **Mức 2 (vừa):** Bài 2: các chỉ số có đúng 2 bit `1` là `3 (011)`, `5 (101)`, `6 (110)`; tổng `nums[3] + nums[5] + nums[6] = 1 + 9 + 2 = 12`. Bài 3: hai dòng đã ghi trong comment. Bài 4: `count = 0`, `while x: count += x & 1; x >>= 1`.
- **Mức 3 (sâu):** Bài 5(a): khi `n = 2^b`, các chỉ số `0..n-1` là mọi chuỗi `b` bit, nên số chỉ số có `k` bit `1` là `comb(b, k)`. Bài 5(b): `sum(v for i, v in enumerate(nums) if i.bit_count() % 2 == 0)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert bin(6) == "0b110" and bin(6).count("1") == 2
assert 6 & 1 == 0 and 6 >> 1 == 3
assert [i for i in range(8) if bin(i).count("1") == 2] == [3, 5, 6]

# Bài 2
assert Solution().sumIndicesWithKSetBits([3, 1, 4, 1, 5, 9, 2, 6], 2) == 1 + 9 + 2

# Bài 3
def sum_todo_ans(nums, k):
    n = len(nums)
    pc = [0] * n
    for i in range(1, n):
        pc[i] = pc[i >> 1] + (i & 1)
    return sum(nums[i] for i in range(n) if pc[i] == k)

for nums, k, expected in examples:
    assert sum_todo_ans(nums, k) == expected

# Bài 4
def popcount_loop(x):
    count = 0
    while x:
        count += x & 1
        x >>= 1
    return count

def sum_loop2(nums, k):
    return sum(nums[i] for i in range(len(nums)) if popcount_loop(i) == k)

for nums, k, expected in examples:
    assert sum_loop2(nums, k) == expected

# Bài 5(a)
from math import comb

def count_indices(n, k):
    b = n.bit_length() - 1              # n = 2^b
    assert n == 1 << b, "n phải là lũy thừa của 2"
    return comb(b, k)

for n in (1, 2, 4, 8, 16, 64, 256):
    for k in range(0, 10):
        assert count_indices(n, k) == sum(1 for i in range(n) if i.bit_count() == k)

# Bài 5(b)
def sum_even_popcount(nums):
    return sum(v for i, v in enumerate(nums) if i.bit_count() % 2 == 0)

assert sum_even_popcount([1, 2, 3, 4, 5, 6, 7, 8]) == 1 + 4 + 6 + 7      # chỉ số 0, 3, 5, 6
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cộng giá trị theo popcount của chỉ số)

```python
from typing import List

class Solution:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        return sum(v for i, v in enumerate(nums) if i.bit_count() == k)
```

**Checklist**

- [ ] Đếm bit của **chỉ số `i`**, không phải của giá trị `nums[i]`?
- [ ] Điều kiện là `== k` (đúng `k` bit), không phải `>= k`?
- [ ] `enumerate(nums)` bắt đầu từ `0` (không `start=1`)?
- [ ] Nếu dùng `bin(i)`: đếm `'1'` (tiền tố `'0b'` không gây sai), không đếm `'0'`?
- [ ] Vòng `while` đếm bit đã cập nhật `i` (`>>=` hoặc `&= i - 1`)?
- [ ] Đã thử: `k = 0` (chỉ `nums[0]`), `k` quá lớn (đáp án `0`), mảng một phần tử?
- [ ] Skeleton đúng `class Solution` / `def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int`?